In [0]:
# Databricks notebook source
# DBTITLE 1,Initialize Parameters
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType
import sys

source_table = "workspace.bronze.grocery_prices"
target_table = "workspace.silver.grocery_prices"

# DBTITLE 2,Read and Transform Ingested Data
try:
    print(f"Reading from Bronze Layer: {source_table}")
    bronze_df = spark.read.table(source_table)

    # Transform strings into strict types, clean strings, and filter scope
    silver_df = (bronze_df
                 # 1. Geographic Scope Isolation (Case-insensitive matching)
                 .filter(F.lower(F.col("GEO")).isin("canada", "ontario"))
                 
                 # 2. Date parsing (StatCan uses yyyy-MM for REF_DATE)
                 .withColumn("SnapshotDate", F.to_date(F.col("REF_DATE"), "yyyy-MM"))
                 
                 # 3. Clean Text Spaces from Product/Commodity Column
                 .withColumn("ProductName", F.trim(F.col("Products")))
                 
                 # 4. Cast metric value to double precision
                 .withColumn("AveragePrice", F.col("VALUE").cast(DoubleType()))
                 
                 # 5. Bring over required structural metadata
                 .select(
                     "SnapshotDate",
                     F.col("GEO").alias("Geography"),
                     "ProductName",
                     "AveragePrice",
                     "UOM", # Unit of measure (e.g., Kilogram, Litre)
                     "ingestion_timestamp"
                 )
                 # Drop structural anomalies missing record value data
                 .filter(F.col("AveragePrice").isNotNull() & F.col("SnapshotDate").isNotNull()))

    # DBTITLE 3,Write Cleaned Data to Silver Delta Layer
    (silver_df.write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(target_table))
     
    print(f"Successfully committed transformations to {target_table}.")

except Exception as e:
    print(f"TRANSFORMATION ERROR in Silver Pipeline: {str(e)}")
    sys.exit(1)